In [ ]:
import sys
import os
import time

# Resolve project root whether the kernel starts in notebooks/ or the repo root.
PROJECT_ROOT = (
    os.path.abspath(os.path.join(os.getcwd(), ".."))
    if os.path.basename(os.getcwd()) == "notebooks"
    else os.getcwd()
)
SRC_PATH = os.path.join(PROJECT_ROOT, "src")
sys.path.insert(0, SRC_PATH)

from data_preprocessing import run_preprocessing
from eda import run_eda
from model import run_modeling
from forecast import run_forecasting


In [ ]:
def run_pipeline():
    overall_start = time.time()

    print("\n" + "=" * 60)
    print("  WALMART SALES FORECASTING PIPELINE")
    print("=" * 60)

    # Preprocessing
    t0 = time.time()
    df, train, test = run_preprocessing()
    print(f"  Preprocessing : {time.time() - t0:.1f}s")

    # EDA
    t0 = time.time()
    eda = run_eda(df)
    print(f"  EDA           : {time.time() - t0:.1f}s  ({len(eda)} plots)")

    # Modeling
    t0 = time.time()
    rf, rf_metrics, all_stats = run_modeling(df, train, test)
    print(f"  Modeling      : {time.time() - t0:.1f}s")

    # Forecasting
    t0 = time.time()
    forecast_df = run_forecasting(df, rf)
    print(f"  Forecasting   : {time.time() - t0:.1f}s  ({len(forecast_df):,} rows)")

    # Summary
    elapsed = time.time() - overall_start

    print("\n" + "=" * 60)
    print("PIPELINE COMPLETE")
    print("=" * 60)
    print(f"  Total time     : {elapsed:.1f}s")
    print(f"  R^2 Score       : {rf_metrics['r2']:.4f}")
    print(f"  MAPE           : {rf_metrics['mape']:.2f}%")
    print(f"  MAE            : ${rf_metrics['mae']:,.0f}")
    print(f"  Forecast rows  : {len(forecast_df):,}")
    print("=" * 60 + "\n")

    return {"df": df, "model": rf, "metrics": rf_metrics, "forecast": forecast_df}

In [ ]:
results = run_pipeline()